In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser  # ✅ Works with your version

# 🔐 Load API keys
import os
from dotenv import load_dotenv
load_dotenv(dotenv_path=".env")
openai_api_key = os.getenv("OPENAI_API_KEY")

prompt = PromptTemplate.from_template("Translate to French: {text}")
llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)
parser = StrOutputParser()

chain = prompt | llm | parser

response = chain.invoke({"text": "Good morning"})
print(response)

Bonjour


In [2]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser

prompt = PromptTemplate.from_template("List 5 programming languages")
parser = CommaSeparatedListOutputParser()
chain = prompt | llm | parser

response = chain.invoke({})
print(response)  # ['Python', 'Java', 'C++', 'JavaScript', 'Ruby']


['1. Python', '2. Java', '3. C++', '4. JavaScript', '5. Ruby']


# Pydantic and structured output parser needs format instructions

In [11]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class ProductInfo(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price in INR")

parser = PydanticOutputParser(pydantic_object=ProductInfo)

prompt = PromptTemplate(
    template="Extract product name and price from: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm #| parser

response = chain.invoke({
    "text": "The Redmi Note 12 is available for ₹14,999."
})
print(response)

content='{\n  "name": "Redmi Note 12",\n  "price": 14999\n}' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 207, 'total_tokens': 228, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DECWn8bSpJaMmWSNGKMYtixZErb3T', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='run--78532f7d-599e-4d39-9ecd-f934d10272a4-0' usage_metadata={'input_tokens': 207, 'output_tokens': 21, 'total_tokens': 228, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}


In [12]:
response.content

'{\n  "name": "Redmi Note 12",\n  "price": 14999\n}'

In [13]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

class ProductInfo(BaseModel):
    name: str = Field(description="Name of the product")
    price: float = Field(description="Price in INR")

parser = PydanticOutputParser(pydantic_object=ProductInfo)

prompt = PromptTemplate(
    template="Extract product name and price from: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

chain = prompt | llm | parser

response = chain.invoke({
    "text": "The Redmi Note 12 is available for ₹14,999."
})
print(response)

name='Redmi Note 12' price=14999.0


In [6]:
parser.get_format_instructions()

'The output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"name": {"description": "Name of the product", "title": "Name", "type": "string"}, "price": {"description": "Price in INR", "title": "Price", "type": "number"}}, "required": ["name", "price"]}\n```'

# Provides dictionary format output

In [10]:
from langchain.output_parsers import ResponseSchema, StructuredOutputParser

schemas = [
    ResponseSchema(name="company", description="Name of the company"),
    ResponseSchema(name="founder", description="Name of the founder"),
]

parser = StructuredOutputParser.from_response_schemas(schemas)

prompt = PromptTemplate(
    template="Extract company and founder from the text: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | llm | parser

response = chain.invoke({
    "text": "Hope AI was founded by Ramisha Rani in Tamil Nadu."
})
print(response)


{'company': 'Hope AI', 'founder': 'Ramisha Rani'}


# You need partial_variables when you have a variable in your template that:

Never changes between calls
You don't want the user to pass it manually every time
If a variable is the same every single time you call the chain, it's a candidate for partial_variables.



In [14]:
from langchain.output_parsers import ResponseSchema, StructuredOutputParser

schemas = [
    ResponseSchema(name="company", description="Name of the company"),
    ResponseSchema(name="founder", description="Name of the founder"),
]

parser = StructuredOutputParser.from_response_schemas(schemas)

prompt = PromptTemplate(
    template="Extract company and founder from the text: {text}\n{format_instructions}",
    input_variables=["text"],
    partial_variables={"format_instructions": parser.get_format_instructions()}
)

chain = prompt | llm #| parser

response = chain.invoke({
    "text": "Hope AI was founded by Ramisha Rani in Tamil Nadu."
})
print(response.content)

```json
{
	"company": "Hope AI",
	"founder": "Ramisha Rani"
}
```
